<a href="https://colab.research.google.com/github/c0sh1/spotify_colabresearch/blob/main/spotify_colabresearch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch

print(torch.cuda.is_available()) #para saber si estás usando la GPU

True


In [2]:
import pandas as pd
import numpy as np
import plotly.express as px

In [3]:
url_spotify = "https://huggingface.co/datasets/maharshipandya/spotify-tracks-dataset/resolve/main/dataset.csv"

df = pd.read_csv(url_spotify, index_col=0)


generos = ["classical","edm", "heavy-metal", "jazz", "reggaeton"]

df = df.dropna();
df = df[df["track_genre"].isin(generos)]
df = df.drop_duplicates(subset="track_id", keep=False)
df = df.drop_duplicates(subset=["track_name", "artists"])
df = df[df["tempo"] > 0]
df = df[df["duration_ms"] < 600000]

print(df.shape)

(3443, 20)


In [4]:
df.head()

,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
16000,7wrYBASu0OoxoDErd4Edxd,Bombay Jayashri,Rehnaa Hai Terre Dil Mein,Zara Zara,58,298266,False,0.643,0.268,11,-15.073,0,0.0900,0.593,0.000002,0.316,0.620,143.813,4,classical
16002,7JGgKHHDgJCJkQCQxyHHdl,Bombay Jayashri;DJ Aftab,Hindi Slowed Reverb Bollywood Lofi,Zara Zara - Lofi,54,219437,False,0.608,0.638,11,-6.008,0,0.0292,0.581,0.017200,0.448,0.439,140.109,4,classical
16003,3YRj4jmwois2ctPnhwSwFo,Bombay Jayashri,Minnalae,Vaseegara,68,299146,False,0.695,0.293,11,-16.278,0,0.0431,0.596,0.015800,0.132,0.637,143.804,4,classical
16004,3tp3ij9dtY3CacQgd1OvRf,Bombay Jayashri;Swattrex,Hindi LoFi Vibe,Zara Zara - LoFi Chill,59,387716,False,0.583,0.308,7,-18.303,0,0.0465,0.581,0.010600,0.257,0.241,118.226,4,classical
16006,6APAPsBCutqTFJ9OFTbeCa,Bombay Jayashri,Kaakha Kaakha (Original Motion Picture Soundtr...,Ondra Renda,59,306626,False,0.642,0.562,9,-9.304,0,0.0607,0.584,0.000000,0.104,0.671,149.820,4,classical


In [5]:
#Meter 1 si es heavy metal y 0 si no

df["es_metal"] = (df["track_genre"] == "heavy-metal").astype(int)

from sklearn.linear_model import LogisticRegression

X_dos = df[["energy", "acousticness"]]
y_metal = df["es_metal"]


neurona = LogisticRegression()
neurona.fit(X_dos, y_metal)


print("Peso de energy: ", neurona.coef_[0][0])
print("Peso de acousticness: ", neurona.coef_[0][1])
print("Sesgo: ", neurona.intercept_[0])

# Energía tiene un peso positivo para clasificar el heavy metal
# Acustica peso negativo para clasificar el heavy metal

# Sesgo: punto de partida. Si todo fuese 0, tiraría más a que no es heavy metal

Peso de energy:  5.424321318126074
Peso de acousticness:  -9.818868621036223
Sesgo:  -4.117925807143688


In [6]:
#pesos y sumas aprendidos

peso_energia = neurona.coef_[0][0]
peso_acustica = neurona.coef_[0][1]
sesgo = neurona.intercept_[0]

#suma ponderada
z = peso_energia * 0.9 + peso_acustica * 0.05 + sesgo


#activacion sigmoide: convierte z en una probabilidad. porque estampos preguntando si es HeavyMetal o no
probabilidad = 1 / (1 + np.exp(-z))

print("z = ", z)
print("Probabilidad a mano: ", probabilidad)


z =  0.2730199481179678
Probabilidad a mano:  0.5678341470739732


In [7]:
#lo mismo preguntando al modelo

cancion = pd.DataFrame({"energy": [0.9], "acousticness": [0.05]})
print("Probabilidad con sklearn", neurona.predict_proba(cancion)[0][1])

Probabilidad con sklearn 0.5678341470739732


## Una neurona es una regresión logística.

* Estamos creando una neurona. El sigmoide convierte cualquier numero en una probabilidad.

* Si tenemos un número entre 0 y 1 lo convertirá en una probabilidad.


In [8]:
valores_z = np.linspace(-8, 8, 100)
valores_probabilidad = 1 / (1 + np.exp(-valores_z))

fig = px.line(x=valores_z, y=valores_probabilidad)
fig.show()

En la gráfica refleja que si le llega un valor negativo dice no, y si le llega positivo dice si.

In [9]:
from sklearn.model_selection import train_test_split

columnas_modelo = ["danceability", "energy", "key", "loudness", "speechiness", "acousticness", "instrumentalness", "liveness", "valence", "tempo", "duration_ms"]

X = df[columnas_modelo]
y = df["track_genre"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
    )


In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

#el renadom forest de la clase de clasificacion
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

acc_rf = accuracy_score(y_test, rf.predict(X_test))

print("Accuracy Random Forest: ", acc_rf)

Accuracy Random Forest:  0.8388969521044993


In [11]:
from sklearn.neural_network import MLPClassifier

# Red con 1 capa oculta de 100 neuronas
red = MLPClassifier(hidden_layer_sizes=(100,), max_iter=1000, random_state=42)

# SIN NORMALIZAR!!
red.fit(X_train, y_train)

acc_red_sin = accuracy_score(y_test, red.predict(X_test))

print("Accuracy Red sin normalizar: ", acc_red_sin)

Accuracy Red sin normalizar:  0.14368650217706821


In [12]:
# Información de la red neuronal

print(red.n_layers_)
print(red.out_activation_)
print(red.coefs_[0].shape)
print(red.coefs_[1].shape)
print(red.n_iter_)

3
softmax
(11, 100)
(100, 5)
22


In [13]:
# ¿que está prediciendo? Contamos respuestas

print(pd.Series(red.predict(X_test)).value_counts())


jazz    689
Name: count, dtype: int64


#### Hay que encontrar el sweet spot.

* Donde la red neuronal funcione de manera eficiente sin que el cómputo se dispare. (no sería eficiente si no)





In [14]:
from sklearn.preprocessing import StandardScaler

# NORMALIZAMOS
escalador = StandardScaler()
escalador.fit(X_train)

X_train_esc = escalador.transform(X_train)
X_test_esc = escalador.transform(X_test)


In [15]:
red = MLPClassifier(hidden_layer_sizes=(100,), max_iter=1000, random_state=42)

red.fit(X_train_esc, y_train)

acc_red = accuracy_score(y_test, red.predict(X_test_esc))

print("Accuracy Red normalizada: ", acc_red)

Accuracy Red normalizada:  0.8040638606676342


/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning:

Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.



In [16]:
# curva de aprendizaje

fig = px.line(y=red.loss_curve_, title="Curva de aprendizaje")
fig.show()


In [17]:
# y si la hacemos más grande? aprende mejor?

# red neuronal dos capas 256 y luego 128
red_grande = MLPClassifier(hidden_layer_sizes=(256, 128), max_iter=1000, random_state=42)

red_grande.fit(X_train_esc, y_train)

print("Accuracy en TRAIN")
print(accuracy_score(y_train, red_grande.predict(X_train_esc)))

print("Accuracy en TEST")
print(accuracy_score(y_test, red_grande.predict(X_test_esc)))


Accuracy en TRAIN
1.0
Accuracy en TEST
0.795355587808418


Si sale que en TEST (0,79 - 79%) funciona peor que en TRAIN (1.0 - 100%), quiere decir que este está aprendiendo los datos de memoria. Por lo que hay sobreajuste, ya que la red grande memoriza las canciones, por lo que no es capaz de generalizar.

In [18]:
# Frases de críticas de cine (HuggingFace), ya separadas en train y test
url_train = "https://huggingface.co/datasets/cornell-movie-review-data/rotten_tomatoes/resolve/main/train.parquet"
url_test = "https://huggingface.co/datasets/cornell-movie-review-data/rotten_tomatoes/resolve/main/test.parquet"

# parquet = formato de tabla comprimido; se lee igual que un CSV
train = pd.read_parquet(url_train)
test = pd.read_parquet(url_test)

print(train.shape)
print(test.shape)

(8530, 2)
(1066, 2)


In [19]:
train.head()

,text,label
0,the rock is destined to be the 21st century's ...,1
1,"the gorgeously elaborate continuation of "" the...",1
2,effective but too-tepid biopic,1
3,if you sometimes like to go to the movies to h...,1
4,"emerges as something rare , an issue movie tha...",1


In [20]:
print(train["label"].head(3))
print(train["label"].tail(3))

0    1
1    1
2    1
Name: label, dtype: int64
8527    0
8528    0
8529    0
Name: label, dtype: int64


In [21]:
# las barajamos

train = train.sample(frac=1, random_state=42)

#### Como convertir texto en números

In [22]:
from sklearn.feature_extraction.text import CountVectorizer

#ejemplo pequeño para verlo

frases= ["great movie", "boring movie", "great great great"]

ejemplo = CountVectorizer()
tabla = ejemplo.fit_transform(frases)

# La tabla de conteos: filas = frases, columnas = palabras
pd.DataFrame(tabla.toarray(), columns=ejemplo.get_feature_names_out())



,boring,great,movie
0,0,1,1
1,1,0,1
2,0,3,0


In [23]:
# ahora con todas las reseñas: con las 5000 palabras mas frecuentes

contador = CountVectorizer(max_features=5000)

# fit SOLO con train (igual que el escalador): aprende el vocabulario
X_train = contador.fit_transform(train["text"])
X_test = contador.transform(test["text"])

y_train = train["label"]
y_test = test["label"]

print(X_train.shape)

(8530, 5000)


In [24]:
# tarda unos 30 - 40 segundos: 200 arboles con 5000 columnas
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

print("Accuracy Random Forest:", accuracy_score(y_test, rf.predict(X_test)))


Accuracy Random Forest: 0.7298311444652908


In [25]:
# early_stopping=True -> para sola cuando deja de mejorar (lo vemos en E)

red = MLPClassifier(hidden_layer_sizes=(64,), early_stopping=True, random_state=42)
red.fit(X_train, y_train)

print("Accuracy Red neuronal:", accuracy_score(y_test, red.predict(X_test)))


Accuracy Red neuronal: 0.7636022514071295


Ahora gana la red. ¿por qué?  -> Con 5000 columnas y cada palabra aportando un poquito ("great"->suma, "boring"->resta), sumar pesos (lo que hace una neurona) funciona mejor que hacer preguntas de SI/NO (lo que hace un árbol).

KERAS

In [26]:
import keras

# semilla para que salga siempre igual
keras.utils.set_random_seed(42)

# Keras necesita una tabla "normal" no vale comprimida como con CountVectorizer
X_train_k = X_train.toarray();
X_test_k = X_test.toarray();



In [27]:
# La red, capa a capa

red_k = keras.Sequential([
    keras.Input(shape=(5000,)),                  # entrada: 5000 palabras
    keras.layers.Dense(64, activation="relu"),   # capa oculta: 64 neuronas
    keras.layers.Dense(1, activation="sigmoid")  # salida -> 1 neurona -> probabilidad de positiva

])

# Cómo aprende: optimizador adam, error sí/no, y que nos diga el accuracy
red_k.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

red_k.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │       320,064 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 320,129 (1.22 MB)

 Trainable params: 320,129 (1.22 MB)

 Non-trainable params: 0 (0.00 B)

In [28]:
# Entrenamiento de 20 épocas
# validation_split=0.2 --> guardar el 20 % del train para vigilar si generalizar

historia = red_k.fit(X_train_k, y_train, epochs=20, validation_split=0.2, verbose=2)

Epoch 1/20
214/214 - 4s - 19ms/step - accuracy: 0.6928 - loss: 0.5987 - val_accuracy: 0.7474 - val_loss: 0.5246
Epoch 2/20
214/214 - 1s - 6ms/step - accuracy: 0.8398 - loss: 0.3798 - val_accuracy: 0.7585 - val_loss: 0.5230
Epoch 3/20
214/214 - 1s - 5ms/step - accuracy: 0.8990 - loss: 0.2592 - val_accuracy: 0.7427 - val_loss: 0.5795
Epoch 4/20
214/214 - 1s - 4ms/step - accuracy: 0.9383 - loss: 0.1801 - val_accuracy: 0.7397 - val_loss: 0.6582
Epoch 5/20
214/214 - 1s - 6ms/step - accuracy: 0.9651 - loss: 0.1244 - val_accuracy: 0.7351 - val_loss: 0.7420
Epoch 6/20
214/214 - 1s - 4ms/step - accuracy: 0.9798 - loss: 0.0859 - val_accuracy: 0.7280 - val_loss: 0.8274
Epoch 7/20
214/214 - 1s - 4ms/step - accuracy: 0.9878 - loss: 0.0598 - val_accuracy: 0.7268 - val_loss: 0.9144
Epoch 8/20
214/214 - 1s - 4ms/step - accuracy: 0.9928 - loss: 0.0424 - val_accuracy: 0.7251 - val_loss: 1.0001
Epoch 9/20
214/214 - 1s - 4ms/step - accuracy: 0.9962 - loss: 0.0308 - val_accuracy: 0.7239 - val_loss: 1.0836


In [29]:
# Si el accuracy sube y el val_loss empieza a bajar, significa que las cosas se las está aprendiendo de memoria y al final no generalizaría.

# Mirar que baje la pérdida. Si la pérdida baja va todo bien. Si de repente sube, indica que hay sobreajuste, se para y coge el mejor anterior.

In [30]:
# El historial de cada época, en un tabla
curvas = pd.DataFrame(historia.history)
curvas["epoca"] = curvas.index + 1

fig = px.line(curvas, x="epoca", y=["accuracy", "val_accuracy"], title="Train sube casi a 100% pero la validación se estanca y baja")

fig.show()


In [31]:
perdida, acierto = red_k.evaluate(X_test_k, y_test, verbose=0)

print("perdida", perdida)
print("acierto", acierto)

perdida 1.5188276767730713
acierto 0.7335835099220276


Como frenar el Overfitting o sobre ajuste ---> EarlyStopping. ----> Dropout

EarlyStopping -----> Vigila el val_loss (error de validacion) y para cuando deja de bajar quedándose con la mejor época de entrenamiento.

###### Si el accuracy sube y el val_loss empieza a bajar, significa que las cosas se las está aprendiendo de memoria y al final no generalizaría.

###### Mirar que baje la pérdida. Si la pérdida baja va todo bien. Si de repente sube, indica que hay sobreajuste, se para y coge el mejor anterior.)

Dropout ---->  en cada paso "apaga" al azar la mitad de las neuronas -----> la red no puede depender de ninguna en concreto ----> memoriza menos. **Como estudiar en grupo sin que siempre conteste el mismo**

In [32]:
keras.utils.set_random_seed(42)

red_k2 = keras.Sequential([
    keras.Input(shape=(5000,)),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dropout(0.5),                    # apaga el 50 % de neuronas en cada paso. DROPOUT
    keras.layers.Dense(1, activation="sigmoid"),

])

red_k2.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

# Para si val_loss no mejora en 2 épocas seguidas y vuelve a la mejor
parada = keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True) #patience = 1 o 2. con el 2, sifgnifica si en 2 épocas seguidas no mejora, te vuelves a la mejor.

historia2 = red_k2.fit(X_train_k, y_train, epochs=50, validation_split=0.2, verbose=2, callbacks=[parada])


Epoch 1/50
214/214 - 4s - 18ms/step - accuracy: 0.6545 - loss: 0.6345 - val_accuracy: 0.7181 - val_loss: 0.5671
Epoch 2/50
214/214 - 1s - 4ms/step - accuracy: 0.8014 - loss: 0.4602 - val_accuracy: 0.7538 - val_loss: 0.5151
Epoch 3/50
214/214 - 1s - 4ms/step - accuracy: 0.8624 - loss: 0.3407 - val_accuracy: 0.7614 - val_loss: 0.5208
Epoch 4/50
214/214 - 1s - 4ms/step - accuracy: 0.8982 - loss: 0.2665 - val_accuracy: 0.7526 - val_loss: 0.5476


In [33]:
print("Épocas que ha entrenado:", len(historia2.history["loss"]))

perdida, acierto = red_k2.evaluate(X_test_k, y_test, verbose=0)

print("Perdida: ", perdida)
print("Acierto: ", acierto)

Épocas que ha entrenado: 4
Perdida:  0.497979998588562
Acierto:  0.7739211916923523


In [34]:
# Reseñas inventadas

mis_frases = [
    "A masterpiece, beautiful and movin",
    "disgusting",
    "Very heavy the final.i couldn't imagine that",
    "Very interesting",
    "I don't like it. 4/10"
]

# Mismo contador de palabras que en el entrenamiento
X_mias = contador.transform(mis_frases).toarray()

probabilidades = red_k2.predict(X_mias, verbose=0)


# Tabla: probabilidad de que cada frase sea positiva
tabla_frases = pd.DataFrame(probabilidades, columns=["positiva"])
tabla_frases["frase"] = mis_frases
tabla_frases.round(2)


,positiva,frase
0,0.86,"A masterpiece, beautiful and movin"
1,0.33,disgusting
2,0.20,Very heavy the final.i couldn't imagine that
3,0.40,Very interesting
4,0.44,I don't like it. 4/10


In [35]:
# modelo "transparente"
logistica_txt = LogisticRegression(max_iter=2000)
logistica_txt.fit(X_train, y_train)
print("Accuracy regresion logística:", accuracy_score(y_test, logistica_txt.predict(X_test)))

# Tiene UN peso por palabra: lo ponemos en una tabla
pesos_palabras = pd.DataFrame({
  "palabra": contador.get_feature_names_out(),
  "peso": logistica_txt.coef_[0],
})

pesos_palabras = pesos_palabras.sort_values("peso")

# Las 10 palabras más negativas y las 10 más positivas
extremos = pd.concat([pesos_palabras.head(10), pesos_palabras.tail(10)])

fig = px.bar(extremos, x="peso", y="palabra", orientation="h",
             title="Transparente: cada palabra suma o resta, y se ve cuánto")
fig.show()


Accuracy regresion logística: 0.7551594746716698


In [39]:
# Pesos de la primera capa de la red: una fila por palabra, una columna por neurona

pesos_red = red_k2.layers[0].get_weights() [0]
print(pesos_red.shape)

# La fila de la palabra "great" (vocabulary_ dice en qué fila está cada palabra)

pesos_great = pesos_red[contador.vocabulary_ ["great"]]
print("Pesos positivos: ", (pesos_great > 0).sum())
print("Pesos negativos: ", (pesos_great < 0).sum())

fig = px.bar(y=pesos_great, title="La palabra 'great' dentro de la red: 64 pesos, unos suman y otros restan")
fig.show()

(5000, 64)
Pesos positivos:  28
Pesos negativos:  36
